# Dataset Sampling Notebook

This notebook provides functionality to sample datasets (e.g., usercommitments) with various time-based and size-based filters.

## Features
- Sample datasets by record count (10K, 100K, 1M records)
- Sample datasets by time periods (24h, 48h, 1 week, 2 weeks, 1 month, 2 months, 3 months, 6 months, 1 year)
- Convert nanosecond timestamps to time differences
- Export sampled datasets to various formats


In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import json
import os
from typing import Dict, List, Any, Optional, Union
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set up plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")


## Configuration

Configure the dataset parameters here:


In [ ]:
# Configuration
CONFIG = {
    # Dataset configuration
    'dataset_name': 'usercommitments',  # Change this to your dataset name
    'timestamp_column': 'requestedAtTimestamp',  # Column containing nanosecond timestamps
    
    # Output configuration
    'output_dir': './sampled_datasets',
    'export_formats': ['json', 'csv', 'parquet'],
    
    # Sampling configuration
    'size_samples': [10000, 100000, 1000000],  # Record counts
    'time_periods': {
        '24h': 24 * 60 * 60 * 1000000000,  # nanoseconds
        '48h': 48 * 60 * 60 * 1000000000,
        '1week': 7 * 24 * 60 * 60 * 1000000000,
        '2weeks': 14 * 24 * 60 * 60 * 1000000000,
        '1month': 30 * 24 * 60 * 60 * 1000000000,
        '2months': 60 * 24 * 60 * 60 * 1000000000,
        '3months': 90 * 24 * 60 * 60 * 1000000000,
        '6months': 180 * 24 * 60 * 60 * 1000000000,
        '1year': 365 * 24 * 60 * 60 * 1000000000
    }
}

# Create output directory
os.makedirs(CONFIG['output_dir'], exist_ok=True)

print(f"Configuration loaded for dataset: {CONFIG['dataset_name']}")
print(f"Timestamp column: {CONFIG['timestamp_column']}")
print(f"Output directory: {CONFIG['output_dir']}")


## Utility Functions

Helper functions for timestamp conversion and data processing:


In [ ]:
def nanoseconds_to_datetime(nanoseconds: Union[int, float]) -> datetime:
    """Convert nanoseconds since epoch to datetime object."""
    return datetime.fromtimestamp(nanoseconds / 1_000_000_000)

def datetime_to_nanoseconds(dt: datetime) -> int:
    """Convert datetime object to nanoseconds since epoch."""
    return int(dt.timestamp() * 1_000_000_000)

def get_current_time_nanoseconds() -> int:
    """Get current time in nanoseconds since epoch."""
    return datetime_to_nanoseconds(datetime.now())

def filter_by_time_period(df: pd.DataFrame, period_nanoseconds: int, 
                         timestamp_col: str = 'requestedAtTimestamp') -> pd.DataFrame:
    """Filter dataframe to include only records within the specified time period."""
    current_time = get_current_time_nanoseconds()
    cutoff_time = current_time - period_nanoseconds
    
    return df[df[timestamp_col] >= cutoff_time].copy()

def sample_by_size(df: pd.DataFrame, size: int, random_state: int = 42) -> pd.DataFrame:
    """Sample dataframe to the specified number of records."""
    if len(df) <= size:
        return df.copy()
    
    return df.sample(n=size, random_state=random_state).copy()

def get_dataset_stats(df: pd.DataFrame, timestamp_col: str = 'requestedAtTimestamp') -> Dict[str, Any]:
    """Get comprehensive statistics about the dataset."""
    if df.empty:
        return {
            'total_records': 0,
            'date_range': None,
            'time_span_days': 0
        }
    
    # Convert timestamps to datetime for analysis
    timestamps = df[timestamp_col].apply(nanoseconds_to_datetime)
    
    return {
        'total_records': len(df),
        'earliest_record': timestamps.min(),
        'latest_record': timestamps.max(),
        'date_range': f"{timestamps.min().strftime('%Y-%m-%d %H:%M:%S')} to {timestamps.max().strftime('%Y-%m-%d %H:%M:%S')}",
        'time_span_days': (timestamps.max() - timestamps.min()).days,
        'time_span_hours': (timestamps.max() - timestamps.min()).total_seconds() / 3600
    }

def export_dataset(df: pd.DataFrame, filename: str, formats: List[str] = None) -> Dict[str, str]:
    """Export dataset to multiple formats."""
    if formats is None:
        formats = CONFIG['export_formats']
    
    exported_files = {}
    
    for format_type in formats:
        filepath = os.path.join(CONFIG['output_dir'], f"{filename}.{format_type}")
        
        if format_type == 'json':
            df.to_json(filepath, orient='records', indent=2)
        elif format_type == 'csv':
            df.to_csv(filepath, index=False)
        elif format_type == 'parquet':
            df.to_parquet(filepath, index=False)
        
        exported_files[format_type] = filepath
    
    return exported_files

print("Utility functions loaded successfully!")
